# Chapter 4.3 가치 반복, 수렴 증명, 그리고 모델 기반이라는 전제 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter04_3_value_iteration_banach.ipynb)

책 본문: [Chapter 4](https://smhanlab.com/book-ml/kor/ml2/chapter04.html)

이 노트북은 4.3절의 가치 반복(value iteration) 코드를 실행하고, 본문의
주장 세 가지를 숫자로 확인합니다: (1) \(T^*\)를 k번 적용하는 것은
"k스텝만 내다보는 것"이며, (2) 정책반복/가치반복은 평가 sweep 수 k를
변수로 하는 같은 스펙트럼의 양 끝단이고, (3) 오차는 매 스텝 최소
\(\gamma\)배로 줄어든다(바나흐 고정점 정리).

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
import numpy as np

## 1. 5칸 GridWorld (4.1/4.2와 같은 환경)

칸 0~4가 일렬로 있고 칸 4가 목표. 각 칸에서 **왼쪽**(action 0) 또는
**오른쪽**(action 1)을 고르고, 걸음마다 보상 \(-1\), 목표에 도착하면
보상 0으로 머문다(absorbing).

In [2]:
# P[s][a] = [(prob, next_state), ...],  R[s][a] = 즉시 보상
n_actions = 2
P = [
    [[(1.0, 0)], [(1.0, 1)]],   # state 0: 왼쪽 self, 오른쪽 -> 1
    [[(1.0, 0)], [(1.0, 2)]],   # state 1
    [[(1.0, 1)], [(1.0, 3)]],   # state 2
    [[(1.0, 2)], [(1.0, 4)]],   # state 3
    [[(1.0, 4)], [(1.0, 4)]],   # state 4 (목표): 보상 0으로 머무름
]
R = [
    [-1, -1], [-1, -1], [-1, -1], [-1, -1], [0, 0],
]
gamma = 0.9

## 2. 가치 반복 (본문 4.3 코드 그대로)

4.1의 `policy_evaluation`과 다른 것은 한 줄: `a = policy[s]` 대신
`max_a`를 취한다. 반복 횟수까지 같이 세본다.

In [3]:
def value_iteration(P, R, n_actions, gamma, theta=1e-6):
    n_states = len(P)
    V = [0.0] * n_states
    iters = 0
    while True:
        delta = 0
        for s in range(n_states):
            v_new = max(R[s][a] + gamma * sum(prob * V[next_s] for prob, next_s in P[s][a])
                        for a in range(n_actions))
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        iters += 1
        if delta < theta:
            break
    policy = [max(range(n_actions),
                  key=lambda a: R[s][a] + gamma * sum(prob * V[next_s] for prob, next_s in P[s][a]))
              for s in range(n_states)]
    return V, policy, iters

V_vi, policy_vi, iters_vi = value_iteration(P, R, n_actions, gamma)
print(f"V = {[round(v, 3) for v in V_vi]}")
print(f"policy = {policy_vi}   (0=왼쪽, 1=오른쪽)")
print(f"반복 횟수: {iters_vi}")

V = [-3.439, -2.71, -1.9, -1.0, 0.0]
policy = [1, 1, 1, 1, 0]   (0=왼쪽, 1=오른쪽)
반복 횟수: 5


## 3. \(T^*\)를 k번 적용 = "k스텝만 내다보기"

\(V_0 = 0\)에서 시작해 최적 연산자 \(T^*\)를 한 sweep씩 적용한다
(synchronous: 다 계산한 뒤 갱신). "보이는 지평선"이 매 스텝 한 칸씩
넓어지고, 목표까지의 최대 거리(4)를 k가 도달하면 값이 더 이상
변하지 않는다 — 본문 4.3의 표를 그대로 만든다.

In [4]:
def T_star(V, P, R, n_actions, gamma):
    """최적 연산자 T*를 한 번 적용 (synchronous)."""
    return [max(R[s][a] + gamma * sum(prob * V[next_s] for prob, next_s in P[s][a])
                for a in range(n_actions)) for s in range(len(P))]

V = [0.0] * 5
print("| k | " + " | ".join(f"V({s})" for s in range(5)) + " |")
print("|---|" + "|---|" * 5)
for k in range(6):
    print(f"| {k} | " + " | ".join(f"{v:.3f}" for v in V) + " |")
    V = T_star(V, P, R, n_actions, gamma)

| k | V(0) | V(1) | V(2) | V(3) | V(4) |
|---||---||---||---||---||---|
| 0 | 0.000 | 0.000 | 0.000 | 0.000 | 0.000 |
| 1 | -1.000 | -1.000 | -1.000 | -1.000 | 0.000 |
| 2 | -1.900 | -1.900 | -1.900 | -1.000 | 0.000 |
| 3 | -2.710 | -2.710 | -1.900 | -1.000 | 0.000 |
| 4 | -3.439 | -2.710 | -1.900 | -1.000 | 0.000 |
| 5 | -3.439 | -2.710 | -1.900 | -1.000 | 0.000 |


## 4. 정책반복/가치반복은 같은 스펙트럼의 두 극한

**수정된 정책 반복**: 정책평고를 sweep k번만 돌린 뒤 바로 정책개선.
\(k=1\)이 가치반복 행, \(k=\infty\)(완전 수렴)가 정책반복 행이다.
초기 정책은 "무조건 오른쪽"([1,1,1,1,1]), V는 sweep마다 이어서 쓰는
warm start (k=∞는 표준 정책평가처럼 매 평가마다 V를 0으로 초기화).
본문 4.3의 표를 재현한다 — k가 달라도 **모두 같은 \(V^*\)**에 도달한다.

In [5]:
def policy_improve(V, P, R, n_actions, gamma):
    return [max(range(n_actions),
                key=lambda a: R[s][a] + gamma * sum(prob * V[next_s] for prob, next_s in P[s][a]))
            for s in range(len(P))]

def evaluate_k(V, policy, P, R, gamma, k, theta=1e-6):
    """정책평가 sweep를 k번 수행 (k='inf'는 수렴할 때까지). (V, sweep 수) 반환."""
    n_states = len(P)
    sweeps = 0
    while True:
        delta = 0
        for s in range(n_states):
            a = policy[s]
            v_new = R[s][a] + gamma * sum(prob * V[next_s] for prob, next_s in P[s][a])
            delta = max(delta, abs(v_new - V[s]))
            V[s] = v_new
        sweeps += 1
        if k == "inf" and delta < theta:
            break
        if k != "inf" and sweeps >= k:
            break
    return V, sweeps

def modified_policy_iteration(P, R, n_actions, gamma, k, init_policy, warm_start=True):
    """k sweep 평가 -> 정책개선 반복. k='inf'가 정책반복."""
    policy = list(init_policy)
    V = [0.0] * len(P)
    outer, total_sweeps = 0, 0
    while True:
        if not warm_start and outer > 0:
            V = [0.0] * len(P)  # 표준 정책평가: 매 평가마다 V 재초기화
        V, sw = evaluate_k(V, policy, P, R, gamma, k)
        total_sweeps += sw
        outer += 1
        new_policy = policy_improve(V, P, R, n_actions, gamma)
        if new_policy == policy:
            return V, policy, outer, total_sweeps
        policy = new_policy

names = {1: "가치 반복", 2: "수정된 정책반복", 4: "수정된 정책반복", "inf": "정책 반복"}
print("| k | 이름 | 바깥 반복 | 총 sweep | 결과 V(0) |")
print("|---|---|---|---|---|")
for k in [1, 2, 4, "inf"]:
    warm = k != "inf"
    V, policy, outer, sweeps = modified_policy_iteration(P, R, n_actions, gamma, k, [1, 1, 1, 1, 1], warm)
    print(f"| {k} | {names[k]} | {outer} | {sweeps} | {V[0]:.3f} |")

| k | 이름 | 바깥 반복 | 총 sweep | 결과 V(0) |
|---|---|---|---|---|
| 1 | 가치 반복 | 5 | 5 | -3.439 |
| 2 | 수정된 정책반복 | 4 | 8 | -3.439 |
| 4 | 수정된 정책반복 | 2 | 8 | -3.439 |
| inf | 정책 반복 | 2 | 10 | -3.439 |


## 5. 바나흐 고정점: 오차가 매 스텝 최소 \(\gamma\)배로 줄어든다

확정적 세계는 유한 스텝에 오차가 정확히 0이 돼 로그 그림이 못 보이기
때문에 **확률적** 변형을 쓴다: 의도한 방향이 0.9 확률로 성공하고 0.1로
자리에 머문다(목표에 도달하면 머무름). 이번에는 오차가 정확히 0이 아닌
채 기하급수적으로 줄어, 실측 오차(파랑)가 바나흐 상한
\(\gamma^k \lVert V_0 - V^* \rVert_\infty\) (검정) 아래를 따라가는 것을
직접 볼 수 있다 — \(T^*\)가 \(\gamma\)-축소 사상임을 숫자로 확인.

In [6]:
p = 0.9  # 의도한 방향으로 갈 확률
P_s, R_s = [], []
for s in range(5):
    if s == 4:
        P_s.append([[(1.0, 4)], [(1.0, 4)]])
        R_s.append([0.0, 0.0])
    else:
        left = s - 1 if s > 0 else 0
        right = s + 1 if s < 4 else 4
        P_s.append([[(p, left), (1 - p, s)], [(p, right), (1 - p, s)]])
        R_s.append([-1.0, -1.0])

# V*는 synchronous로 계산(아래에서 측정할 T* 반복과 같은 연산).
# in-place value_iteration은 delta<1e-6에서 멈추며, 그 정지점이 진한 고정점과
# 약 1e-8 벌어져 있어 꼬리 구간 감소비가 그 노이즈 플로어에 의해 1을 넘을 수 있다.
V_star = [0.0] * 5
while True:
    V_next = T_star(V_star, P_s, R_s, n_actions, gamma)
    if max(abs(a - b) for a, b in zip(V_star, V_next)) < 1e-13:
        V_star = V_next
        break
    V_star = V_next
print(f"확률적 세계 V* = {[round(v, 3) for v in V_star]}")

# synchronous 가치 반복 추적: 실측 오차 vs 바나흐 상한
V = [0.0] * 5
K = 40
scale = max(abs(v) for v in V_star)  # ||V_0 - V*||_inf (V_0 = 0)
errs, bounds = [], []
for k in range(K):
    errs.append(max(abs(V[s] - V_star[s]) for s in range(5)))
    bounds.append((gamma ** k) * scale)
    V = T_star(V, P_s, R_s, n_actions, gamma)

# 감소비는 부동소수점 노이즈 플로어(약 1e-14) 위 구간에서만 측정: 그 아래는
# 오차가 소수점 아래에서 왕복(fluctuate)하므로 감소율 확인에 의미가 없다.
floor = 1e-10 * scale
ratios = [errs[k + 1] / errs[k] for k in range(K - 1) if errs[k] > floor]
print(f"실측 감소비 err(k+1)/err(k)의 최대치 = {max(ratios):.4f}  (gamma = {gamma})")
print(f"바나흐 상한 err(k) <= gamma^k*||V_0 - V*||_inf (모든 k): "
      f"{all(e <= b for e, b in zip(errs, bounds))}")

확률적 세계 V* = [-3.723, -2.948, -2.077, -1.099, 0.0]
실측 감소비 err(k+1)/err(k)의 최대치 = 0.7314  (gamma = 0.9)
바나흐 상한 err(k) <= gamma^k*||V_0 - V*||_inf (모든 k): True


In [7]:
# 본문 4.3 삽입 그림: 좌 = 지평선이 넓어지는 과정(확정적), 우 = 바나흐 상한(확률적)
V = [0.0] * 5
traj = [V[:]]
for k in range(1, 9):
    V = T_star(V, P, R, n_actions, gamma)
    traj.append(V[:])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
for s in range(5):
    ax.plot(range(9), [t[s] for t in traj], marker="o", ms=4, label=f"state {s}")
ax.set_xlabel("k (T* 적용 횟수)")
ax.set_ylabel("V_k(s)")
ax.set_title("지평선이 한 칸씩 넓어진다 (목표: 칸 4)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

ax = axes[1]
ax.semilogy(range(K), [max(e, 1e-16) for e in errs], "--o", ms=3,
            color="tab:blue", label="실측 ||V_k - V*||_inf")
ax.semilogy(range(K), bounds, "--", color="black",
            label=r"바나흐 상한 $\gamma^k\,||V_0 - V^*||_\infty$")
ax.set_xlabel("k (반복 횟수)")
ax.set_ylabel("최대 오차 (로그축)")
ax.set_title(f"오차가 매 스텝 최소 {gamma}배로 줄어든다")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

fig.tight_layout()
IMG = "/home/smhan/book-ml/kor/src/images"
fig.savefig(IMG + "/ch04_3_value_iteration_horizon.svg", bbox_inches="tight")
print(f"저장: {IMG}/ch04_3_value_iteration_horizon.svg")

저장: /home/smhan/book-ml/kor/src/images/ch04_3_value_iteration_horizon.svg
